# VolatilityLab — Feature 2: Implied Volatility Explorer

**Submission-ready research prototype.** Download current option chains for a selected ticker, screen quote quality, explore call/put surfaces and an out-of-the-money (OTM) volatility skew, and review data-quality diagnostics.

Run cells **top to bottom** in Google Colab. Options data are live/delayed and results will change. Interpolated surfaces are **exploratory**, not arbitrage-free option-pricing models. The final quality-screened plot displays observed contracts alongside interpolation.

In [ ]:
!pip install -q yfinance plotly pandas numpy scipy

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from scipy.interpolate import griddata
from IPython.display import display

# Change this ticker and rerun the cells below.
ticker = "VOO"

# Exploratory quote-quality thresholds
MIN_BID = 0
MAX_SPREAD = 0.30
STRICT_SPREAD = 0.20
MAX_TRADE_AGE_DAYS = 7
MIN_CONTRACTS_PER_EXPIRY = 5


## 1. Retrieve and screen option chains

Both calls and puts are collected across available expirations. Screening requires positive bid, ask greater than bid, plausible reported implied volatility, and a relative spread below 30%. The current stock price is used to compute **moneyness = strike / spot**. A stricter subset is retained for the final plot.

In [ ]:
def load_option_data(ticker):
    stock = yf.Ticker(ticker)
    history = stock.history(period="5d")
    if history.empty or history["Close"].dropna().empty:
        raise ValueError(f"No stock prices available for {ticker}")
    spot = float(history["Close"].dropna().iloc[-1])
    expirations = stock.options
    if not expirations:
        raise ValueError(f"No option expirations available for {ticker}")

    chunks = []
    for expiration in expirations:
        try:
            chain = stock.option_chain(expiration)
            for kind, contracts in (("Call", chain.calls), ("Put", chain.puts)):
                if not contracts.empty:
                    part = contracts.copy()
                    part["optionType"] = kind
                    part["expiration"] = expiration
                    chunks.append(part)
        except Exception as exc:
            print(f"Skipped {expiration}: {exc}")
    if not chunks:
        raise ValueError("No option chains could be downloaded")

    raw = pd.concat(chunks, ignore_index=True)
    options = raw.copy()
    for col in ("bid", "ask", "strike", "impliedVolatility"):
        options[col] = pd.to_numeric(options[col], errors="coerce")
    options = options.loc[
        (options["bid"] > MIN_BID) &
        (options["ask"] > options["bid"]) &
        (options["impliedVolatility"].between(0.01, 3.0))
    ].copy()
    options["relativeSpread"] = (
        (options["ask"] - options["bid"]) /
        ((options["ask"] + options["bid"]) / 2)
    )
    options = options.loc[options["relativeSpread"] < MAX_SPREAD].copy()
    options["expiration"] = pd.to_datetime(options["expiration"])
    today = pd.Timestamp.now(tz="America/New_York").normalize().tz_localize(None)
    options["daysToExpiration"] = (options["expiration"] - today).dt.days
    options["Moneyness"] = options["strike"] / spot
    options["IV_Percent"] = options["impliedVolatility"] * 100
    options = options.loc[
        options["daysToExpiration"].between(7, 365) &
        options["Moneyness"].between(0.70, 1.30) &
        options["IV_Percent"].between(5, 150)
    ].copy()
    if options.empty:
        raise ValueError("No contracts passed the initial filters")
    return spot, raw, options

spot, raw_options, options = load_option_data(ticker)
print(f"{ticker} spot: ${spot:.2f} | raw contracts: {len(raw_options)} | screened: {len(options)}")
print("Screened expirations:", options["expiration"].nunique())
display(options[["optionType", "strike", "expiration", "daysToExpiration", "IV_Percent", "relativeSpread"]].head())


## 2. Explore separate call and put volatility surfaces

These surfaces use linear interpolation only inside the observed points' convex hull; blank areas are intentional. Individual call and put surfaces can be selected from the dropdown.

In [ ]:
def make_surface(df, x_col="strike", title="Implied Volatility Surface", min_points=10):
    points = df.groupby([x_col, "daysToExpiration"], as_index=False)["IV_Percent"].median()
    if (len(points) < min_points or points[x_col].nunique() < 2
            or points["daysToExpiration"].nunique() < 2):
        return None
    X, Y = np.meshgrid(
        np.linspace(points[x_col].min(), points[x_col].max(), 60),
        np.linspace(points["daysToExpiration"].min(), points["daysToExpiration"].max(), 60)
    )
    try:
        Z = griddata((points[x_col], points["daysToExpiration"]),
                     points["IV_Percent"], (X, Y), method="linear")
    except Exception:
        return None
    if np.isnan(Z).all():
        return None
    return X, Y, Z, points

fig = go.Figure()
labels = []
for kind in ("Call", "Put"):
    result = make_surface(options.loc[options["optionType"] == kind])
    if result is None:
        print(f"Not enough points for {kind} surface")
        continue
    X, Y, Z, _ = result
    fig.add_trace(go.Surface(x=X, y=Y, z=Z, name=kind,
                             visible=(len(labels) == 0), colorscale="Viridis"))
    labels.append(kind)
if not labels:
    print("No call/put surface can be interpolated with available quotes.")
else:
    buttons = [dict(label=kind + "s", method="update",
                    args=[{"visible": [j == i for j in range(len(labels))]}])
               for i, kind in enumerate(labels)]
    fig.update_layout(title=f"{ticker} — Call / Put IV Surface",
        scene=dict(xaxis_title="Strike ($)", yaxis_title="Days to expiration",
                   zaxis_title="Implied volatility (%)"),
        updatemenus=[dict(buttons=buttons, direction="down", x=0.05, y=1.10)],
        height=700)
    fig.show()


## 3. OTM skew and volatility smile

Use puts with strikes below spot and calls above spot, focusing on 85%–115% moneyness and 14–365 days. The 2D chart selects the expiration with the most retained contracts.

In [ ]:
clean_surface = options.loc[
    (((options["optionType"] == "Call") & (options["strike"] >= spot)) |
     ((options["optionType"] == "Put") & (options["strike"] <= spot))) &
    options["Moneyness"].between(0.85, 1.15) &
    options["daysToExpiration"].between(14, 365)
].copy()
print("OTM contracts:", len(clean_surface), "| expirations:", clean_surface["expiration"].nunique())
if clean_surface.empty:
    print("No OTM contracts passed the filters.")
else:
    selected_expiration = clean_surface.groupby("expiration").size().idxmax()
    smile = clean_surface.loc[clean_surface["expiration"] == selected_expiration].sort_values("Moneyness")
    fig_smile = px.scatter(smile, x="Moneyness", y="IV_Percent", color="optionType",
                           hover_data=["strike", "bid", "ask"],
                           title=f"{ticker} — OTM IV Skew ({selected_expiration.date()})")
    fig_smile.update_traces(mode="lines+markers")
    fig_smile.add_vline(x=1, line_dash="dash", line_color="gray")
    fig_smile.update_layout(xaxis_title="Strike / Spot", yaxis_title="IV (%)")
    fig_smile.show()


## 4. Quote-quality diagnostics and final quality-filtered surface

`lastTradeDate` measures the age of the last **trade**, not the quote. Strict screening requires a relative spread at most 20% and a last trade no older than seven days. Sparse expirations are omitted. Observed points appear on the final surface so interpolation gaps remain visible.

In [ ]:
quality_data = clean_surface.copy()
quality_data["lastTradeDate"] = pd.to_datetime(quality_data["lastTradeDate"], utc=True, errors="coerce")
now = pd.Timestamp.now(tz="UTC")
quality_data["tradeAgeDays"] = (now - quality_data["lastTradeDate"]).dt.total_seconds() / 86400
quality_data["staleTrade"] = quality_data["tradeAgeDays"] > MAX_TRADE_AGE_DAYS
quality_data["wideSpread"] = quality_data["relativeSpread"] > STRICT_SPREAD
summary = pd.DataFrame({
    "Metric": ["OTM contracts", "Expirations", "Median trade age (days)",
               "Trades older than 7 days (%)", "Spreads above 20% (%)", "Median IV (%)"],
    "Value": [len(quality_data), quality_data["expiration"].nunique(),
              quality_data["tradeAgeDays"].median(), quality_data["staleTrade"].mean()*100,
              quality_data["wideSpread"].mean()*100, quality_data["IV_Percent"].median()]
})
display(summary.round(2))

high_quality_surface = quality_data.loc[
    (quality_data["relativeSpread"] <= STRICT_SPREAD) &
    quality_data["tradeAgeDays"].between(0, MAX_TRADE_AGE_DAYS)
].copy()
counts = high_quality_surface.groupby("expiration").size()
valid_expirations = counts[counts >= MIN_CONTRACTS_PER_EXPIRY].index
final_data = high_quality_surface.loc[high_quality_surface["expiration"].isin(valid_expirations)]
print("Strict contracts:", len(high_quality_surface), "| final contracts:", len(final_data))
display(counts.rename("Contracts").reset_index())

result = make_surface(final_data, x_col="Moneyness")
if result is None:
    print("Not enough high-quality observations for a reliable interpolated surface. Review the observed data instead.")
else:
    X, Y, Z, observed = result
    fig_final = go.Figure()
    fig_final.add_trace(go.Surface(x=X, y=Y, z=Z, colorscale="Viridis", opacity=0.80,
                                   name="Interpolated IV"))
    fig_final.add_trace(go.Scatter3d(x=observed["Moneyness"], y=observed["daysToExpiration"],
                                     z=observed["IV_Percent"], mode="markers",
                                     marker=dict(size=4, color="red"), name="Observed contracts"))
    fig_final.update_layout(title=f"{ticker} — Quality-Filtered OTM IV Surface",
        scene=dict(xaxis_title="Moneyness (strike / spot)", yaxis_title="Days to expiration",
                   zaxis_title="Implied volatility (%)"), height=700)
    fig_final.show()


## Interpretation and limitations

- The **3D surface** illustrates implied volatility variation across strike and expiration; the **2D skew** isolates one expiration.
- The dataset is a current snapshot, not historical options quotes. Yahoo Finance values may be delayed, stale, missing, or inconsistent.
- Screening and linear interpolation are exploratory. They do not ensure no-arbitrage consistency, executable prices, or reliable Greeks.
- `lastTradeDate` is **not** a timestamp for bid/ask quotes. Contracts and expiration coverage vary by ticker and date.
- For a competition demonstration, record the ticker, run date, retained contract counts, and assumptions; do not present the interpolated surface as a calibrated option-pricing model.
